# 00 — Evaluation & Feature Importance Utilities

Shared functions called by all model training notebooks via:
```python
%run /Workspace/Users/daniel.branco@cgi.com/Tese/00_Evaluation
```

### Contents

| Section | Functions |
|---|---|
| Metrics | `evaluate_binary()`, `threshold_sweep()` |
| Plots | `plot_confusion_matrix()`, `plot_roc_curve()`, `plot_pr_curve()` |
| MLflow | `log_evaluation_to_mlflow()` |
| Spark helper | `extract_prob_positive()` |
| Feature importance | `classify_feature()`, `grouped_importance()`, `plot_top_features()`, `plot_grouped_importance()` |

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import pandas as pd

import mlflow
from pyspark.sql import functions as F
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, precision_recall_curve,
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    accuracy_score, matthews_corrcoef, brier_score_loss, log_loss,
)

from pyspark.ml.functions import vector_to_array
from pyspark.ml.feature import VectorAssembler, FeatureHasher

## Spark ML helper

In [0]:
def extract_prob_positive(df, prob_col="probability"):
    """Extract P(class=1) from Spark ML probability vector."""
    return df.withColumn(
        "prob_pos",
        vector_to_array(F.col(prob_col))[1]
    )

## Metric computation

In [0]:
def evaluate_binary(y_true, y_prob, threshold=0.5, title=""):
    """
    Compute the full metric suite used across all models.

    Parameters
    ----------
    y_true     : array of 0/1 labels
    y_prob     : array of predicted probabilities for class 1
    threshold  : decision boundary (default 0.5)
    title      : optional label printed in the summary

    Returns
    -------
    dict with all metrics
    """
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        "threshold":  float(threshold),
        "accuracy":   float(accuracy_score(y_true, y_pred)),
        "precision":  float(precision_score(y_true, y_pred, zero_division=0)),
        "recall":     float(recall_score(y_true, y_pred, zero_division=0)),
        "f1":         float(f1_score(y_true, y_pred, zero_division=0)),
        "mcc":        float(matthews_corrcoef(y_true, y_pred)),
        "AUROC":      float(roc_auc_score(y_true, y_prob)),
        "AUPRC":      float(average_precision_score(y_true, y_prob)),
        "brier":      float(brier_score_loss(y_true, y_prob)),
        "log_loss":   float(log_loss(y_true, y_prob)),
        }
    if title:
        print(f"\n{'═'*60}")
        print(f"  {title}")
        print(f"{'═'*60}")
    for k, v in metrics.items():
        print(f"  {k:<14s}: {v:.4f}")
    return metrics


def threshold_sweep(y_true, y_prob, thresholds=(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99)):
    rows = []
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        rows.append({
            "threshold": float(t),
            "precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "recall":    float(recall_score(y_true, y_pred, zero_division=0)),
            "f1":        float(f1_score(y_true, y_pred, zero_division=0)),
            "mcc":       float(matthews_corrcoef(y_true, y_pred)),
        })
    return rows

## Plotting functions

In [0]:
def plot_confusion_matrix(y_true, y_pred, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred)
    plt.ioff()
    fig, ax = plt.subplots(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(cm, display_labels=["No overload", "Overload"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(title)
    plt.tight_layout()
    plt.ion()
    return fig


def plot_roc_curve(y_true, y_prob, title="ROC Curve"):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val = roc_auc_score(y_true, y_prob)
    plt.ioff()
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, label=f"AUROC = {auc_val:.4f}")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.set_title(title); ax.legend(loc="lower right")
    plt.tight_layout()
    plt.ion()
    return fig


def plot_pr_curve(y_true, y_prob, title="Precision-Recall Curve"):
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    plt.ioff()
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(rec, prec, label=f"AUPRC = {ap:.4f}")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title(title); ax.legend(loc="upper right")
    plt.tight_layout()
    plt.ion()
    return fig

## MLflow logging

In [0]:
def log_evaluation_to_mlflow(metrics, y_true, y_prob, threshold=0.5, prefix="test"):
    """
    Log all metrics, plots, and threshold sweep to the active MLflow run.

    Call this inside a `with mlflow.start_run(...)` block.
    """
    # Scalar metrics
    for k, v in metrics.items():
        mlflow.log_metric(f"{prefix}_{k}", v)

    y_pred = (y_prob >= threshold).astype(int)

    # Confusion matrix
    fig_cm = plot_confusion_matrix(y_true, y_pred, title=f"{prefix} — Confusion Matrix")
    mlflow.log_figure(fig_cm, f"{prefix}_confusion_matrix.png")
    plt.close(fig_cm)

    # ROC curve
    fig_roc = plot_roc_curve(y_true, y_prob, title=f"{prefix} — ROC Curve")
    mlflow.log_figure(fig_roc, f"{prefix}_roc_curve.png")
    plt.close(fig_roc)

    # PR curve
    fig_pr = plot_pr_curve(y_true, y_prob, title=f"{prefix} — PR Curve")
    mlflow.log_figure(fig_pr, f"{prefix}_pr_curve.png")
    plt.close(fig_pr)

    # Threshold sweep table
    sweep = threshold_sweep(y_true, y_prob)
    mlflow.log_table(pd.DataFrame(sweep), artifact_file=f"{prefix}_threshold_sweep.json")

## Feature importance utilities

These require the calling notebook to have defined the feature group constants  
(`WEATHER_RAW_COLS`, `WEATHER_DERIVED_COLS`, `LOAD_RATIO_COLS`, `TEMPORAL_COLS`, `EVENT_COLS`)  
before calling `classify_feature()`.

In [0]:
def classify_feature(fname):
    """
    Map a feature name to its conceptual group.
    Works with both raw column names and OHE-expanded names.

    Requires the following sets to be defined in the calling notebook's scope:
    WEATHER_RAW_SET, WEATHER_DERIVED_SET, LOAD_RATIO_SET, TEMPORAL_SET, EVENT_SET
    """
    if fname.startswith("ID_prefix"):   return "Transformer ID"
    if fname.startswith("CONCELHO"):    return "Concelho"
    if fname in LOAD_RATIO_SET:         return "Load ratio"
    if fname in WEATHER_RAW_SET:        return "Weather (raw)"
    if fname in WEATHER_DERIVED_SET:    return "Weather (derived)"
    if fname in TEMPORAL_SET:           return "Temporal"
    if fname in EVENT_SET:              return "Events"
    if "_lag_" in fname:                return "Lag features"
    if any(w in fname for w in ["_mean_", "_std_", "_max_"]):
        return "Rolling stats"
    return "Other"


def grouped_importance(feat_names, importances):
    """
    Aggregate feature importances by group.

    Parameters
    ----------
    feat_names  : list of feature name strings
    importances : list/array of importance values (|coeff|, gain, etc.)

    Returns
    -------
    dict  {group: {"sum": float, "mean": float, "n_dims": int}}
    """
    groups = defaultdict(lambda: {"sum": 0.0, "values": []})
    for name, imp in zip(feat_names, importances):
        g = classify_feature(name)
        groups[g]["sum"] +=  np.abs(imp)
        groups[g]["values"].append(np.abs(imp))
    return {
        g: {"sum": d["sum"], "mean": d["sum"] / len(d["values"]), "n_dims": len(d["values"])}
        for g, d in groups.items()
    }


def plot_top_features(feat_names, importances, top_n=20, title="Top Features", xlabel="|Coefficient|"):
    """Horizontal bar chart of top individual features."""
    idx = np.argsort(np.abs(importances))[::-1][:top_n]
    names = [feat_names[i] for i in idx]
    vals  = [np.abs(importances[i]) for i in idx]
    fig, ax = plt.subplots(figsize=(8, builtins.max(4, top_n * 0.3)))
    y_pos = range(len(names))
    ax.barh(y_pos, vals[::-1], color="steelblue")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(names[::-1], fontsize=9)
    ax.set_xlabel(xlabel); ax.set_title(title)
    plt.tight_layout()
    return fig


def plot_grouped_importance(grouped_dict, title="Feature Group Importance", xlabel="Sum |Importance|"):
    """Horizontal bar chart of grouped importance."""
    sorted_groups = sorted(grouped_dict.items(), key=lambda x: x[1]["sum"], reverse=True)
    names = [g for g, _ in sorted_groups]
    sums  = [d["sum"] for _, d in sorted_groups]
    dims  = [d["n_dims"] for _, d in sorted_groups]
    fig, ax = plt.subplots(figsize=(8, builtins.max(3, len(names) * 0.5)))
    y_pos = range(len(names))
    ax.barh(y_pos, sums[::-1], color="darkorange")
    ax.set_yticks(y_pos)
    ax.set_yticklabels([f"{n}  (n={d})" for n, d in zip(names[::-1], dims[::-1])], fontsize=10)
    ax.set_xlabel(xlabel); ax.set_title(title)
    plt.tight_layout()
    return fig

In [0]:
print("✅ 00_Evaluation loaded — evaluation, plotting, and feature importance functions available")